In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load datasets
movies = pd.read_csv("/content/DSA4060_Week6_Movies.csv")
ratings = pd.read_csv("/content/DSA4060_Week6_Ratings.csv")

# Inspect shapes and first few rows
print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)

display(movies.head())
display(ratings.head())

Movies shape: (20, 3)
Ratings shape: (120, 4)


,movie_id,title,genres
0,1,Galaxy Mission,Action|Sci-Fi
1,2,Love in Nairobi,Romance|Drama
2,3,Final Battle,Action
3,4,Family Weekend,Comedy|Family
4,5,Future World,Sci-Fi|Action


,user_id,movie_id,rating,timestamp
0,1,2,3,1
1,1,10,3,2
2,1,1,5,3
3,1,9,3,4
4,1,7,3,5


In [3]:
from sklearn.model_selection import train_test_split

# Split ratings into train (80%) and test (20%) sets
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

# Verify the shapes of your splits
print("Training set shape:", train.shape)
print("Testing set shape:", test.shape)

Training set shape: (96, 4)
Testing set shape: (24, 4)


In [4]:
# Aggregate movie statistics from the training set
movie_stats = train.groupby("movie_id").agg(
    mean_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()

# Filter for movies with a minimum number of ratings (e.g., min_ratings = 5) to avoid noise
min_ratings = 5
popular_movies = movie_stats[movie_stats["rating_count"] >= min_ratings]

# Sort by mean rating and rating count to find the top popular movies
top_popular = popular_movies.sort_values(
    ["mean_rating", "rating_count"],
    ascending=[False, False]
).head(10)

print("Top 10 Popular Movies:")
display(top_popular)

Top 10 Popular Movies:


,movie_id,mean_rating,rating_count
5,6,4.000000,5
3,4,3.800000,5
9,10,3.666667,6
6,7,3.600000,5
1,2,3.500000,8
8,9,3.333333,6
18,19,3.200000,5
19,20,3.125000,8
2,3,3.000000,6
13,14,2.857143,7


In [5]:
# Create an item feature matrix from movie genres (using one-hot encoding)
# Assuming 'genres' column exists in movies dataframe (e.g., separated by '|' or similar)
item_genres = movies.set_index("movie_id")["genres"].str.get_dummies(sep="|")

# Alternatively, if genres are already formatted or handled via feature extraction:
item_feature_matrix = item_genres

# Define a function to score a movie for a specific user based on their training ratings
def content_score(user_id, movie_id, train_df):
    # Get user's rated movies and ratings from training set
    user_ratings = train_df[train_df["user_id"] == user_id]
    if user_ratings.empty:
        return 0.0

    # Simple content-based scoring logic using genre similarity or weighted preferences
    # (Or compute cosine similarity between target movie and user profile vector)
    target_genres = item_feature_matrix.loc[movie_id]

    # Calculate weighted average based on user's high ratings (e.g., rating >= 4)
    liked_movies = user_ratings[user_ratings["rating"] >= 4]["movie_id"]
    if liked_movies.empty:
        return 0.0

    liked_vectors = item_feature_matrix.loc[item_feature_matrix.index.isin(liked_movies)]
    user_profile = liked_vectors.mean(axis=0)

    # Compute dot product or cosine similarity between user profile and target movie
    score = np.dot(user_profile, target_genres)
    return score

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Create a user-item rating matrix from the training set (rows: users, columns: movies)
user_item_matrix = train.pivot(index="user_id", columns="movie_id", values="rating").fillna(0)

# 2. Compute item-item cosine similarity matrix
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

# 3. Define a prediction function for item-based collaborative filtering
def item_cf_predict(user_id, movie_id, train_df):
    if movie_id not in item_similarity_df.index:
        return 3.0  # Fallback to a neutral rating if movie is missing from item matrix

    # Get ratings given by this user
    user_ratings = train_df[train_df["user_id"] == user_id]
    if user_ratings.empty:
        return 3.0

    rated_movies = user_ratings["movie_id"].values
    ratings = user_ratings["rating"].values

    # Get similarities between the target movie and movies the user has rated
    sims = item_similarity_df.loc[movie_id, rated_movies].values

    # Compute weighted average of ratings based on item similarity
    if sum(abs(sims)) == 0:
        return ratings.mean() # Fallback if no similarity

    predicted_rating = np.dot(sims, ratings) / sum(abs(sims))
    return predicted_rating

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- 1. Popularity Baseline Evaluation ---
# Calculate movie average ratings from the training set to act as predictions
global_mean = train["rating"].mean()
movie_means = train.groupby("movie_id")["rating"].mean().to_dict()

baseline_predictions = []
for _, row in test.iterrows():
    # Predict the movie's average training rating, or fallback to global mean
    pred = movie_means.get(row["movie_id"], global_mean)
    baseline_predictions.append(pred)

baseline_mae = mean_absolute_error(test["rating"], baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(test["rating"], baseline_predictions))

# --- 2. Content-Based Evaluation ---
content_predictions = []
for _, row in test.iterrows():
    pred = content_score(row["user_id"], row["movie_id"], train)
    # Clip scores to be within the realistic rating scale of [1, 5] if necessary
    pred = np.clip(pred, 1.0, 5.0)
    content_predictions.append(pred)

content_mae = mean_absolute_error(test["rating"], content_predictions)
content_rmse = np.sqrt(mean_squared_error(test["rating"], content_predictions))

# --- 3. Item-Based CF Evaluation ---
test_predictions = []
for _, row in test.iterrows():
    pred = item_cf_predict(row["user_id"], row["movie_id"], train)
    test_predictions.append(pred)

cf_mae = mean_absolute_error(test["rating"], test_predictions)
cf_rmse = np.sqrt(mean_squared_error(test["rating"], test_predictions))

print("Baseline MAE:", round(baseline_mae, 3), "RMSE:", round(baseline_rmse, 3))
print("Content MAE:", round(content_mae, 3), "RMSE:", round(content_rmse, 3))
print("CF MAE:", round(cf_mae, 3), "RMSE:", round(cf_rmse, 3))

# Build the final rating metrics comparison table
rating_results = pd.DataFrame({
    "Model": [
        "Popularity Baseline",
        "Content-Based",
        "Item-Based CF"
    ],
    "MAE": [baseline_mae, content_mae, cf_mae],
    "RMSE": [baseline_rmse, content_rmse, cf_rmse]
})

display(rating_results.sort_values("MAE"))

Baseline MAE: 0.77 RMSE: 0.916
Content MAE: 2.09 RMSE: 2.261
CF MAE: 0.709 RMSE: 0.862


,Model,MAE,RMSE
2,Item-Based CF,0.708795,0.861592
0,Popularity Baseline,0.769841,0.916010
1,Content-Based,2.090278,2.261033


In [9]:
TOP_K = 3

# 1. Define relevance rule (rating >= 4)
test["relevant"] = test["rating"] >= 4

# 2. Get list of all movie IDs
all_movie_ids = movies["movie_id"].tolist()

def unseen_movies(user_id):
    seen = set(train.loc[train["user_id"] == user_id, "movie_id"])
    return [m for m in all_movie_ids if m not in seen]

# 3. Top-N Recommender Functions
def recommend_popularity(user_id, k=TOP_K):
    unseen = set(unseen_movies(user_id))
    return [m for m in popularity_order if m in unseen][:k]

def recommend_content(user_id, k=TOP_K):
    scored = [(m, content_score(user_id, m, train)) for m in unseen_movies(user_id)]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [m for m, s in scored[:k]]

def recommend_cf(user_id, k=TOP_K):
    scored = [(m, item_cf_predict(user_id, m, train)) for m in unseen_movies(user_id)]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [m for m, s in scored[:k]]

In [11]:
# Define the popularity_order variable needed by recommend_popularity
# We sort the movie_stats (or movies) by mean_rating and rating_count
popularity_order = movie_stats.sort_values(
    ["mean_rating", "rating_count"],
    ascending=[False, False]
)["movie_id"].tolist()

# Fallback for any movies that are not in movie_stats (have no training ratings)
unrated_movies = [m for m in all_movie_ids if m not in popularity_order]
popularity_order.extend(unrated_movies)

def precision_recall_at_k(recs, relevant, k):
    recs = recs[:k]
    hits = len(set(recs) & set(relevant))
    precision = hits / k
    recall = hits / len(relevant) if relevant else np.nan
    return precision, recall

def ndcg_at_k(recs, relevant, k):
    dcg = 0.0
    for rank, movie_id in enumerate(recs[:k], start=1):
        if movie_id in relevant:
            dcg += 1 / np.log2(rank + 1)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else np.nan

def evaluate_ranking(recommender, k=TOP_K):
    rows = []
    for user_id in sorted(test["user_id"].unique()):
        relevant = test.loc[
            (test["user_id"] == user_id) & (test["rating" ] >= 4),
            "movie_id"
        ].tolist()
        if not relevant:
            continue
        recs = recommender(user_id, k)
        precision, recall = precision_recall_at_k(recs, relevant, k)
        rows.append({
            "user_id": user_id,
            "Precision@K": precision,
            "Recall@K": recall,
            "NDCG@K": ndcg_at_k(recs, relevant, k)
        })
    result = pd.DataFrame(rows)
    return {
        "Precision@3": result["Precision@K"].mean(),
        "Recall@3": result["Recall@K"].mean(),
        "NDCG@3": result["NDCG@K"].mean()
    }

# Run ranking evaluation for all three models
pop_rank = evaluate_ranking(recommend_popularity, TOP_K)
content_rank = evaluate_ranking(recommend_content, TOP_K)
cf_rank = evaluate_ranking(recommend_cf, TOP_K)

ranking_results = pd.DataFrame([
    {"Model": "Popularity Baseline", **pop_rank},
    {"Model": "Content-Based", **content_rank},
    {"Model": "Item-Based CF", **cf_rank}
])

display(ranking_results)

,Model,Precision@3,Recall@3,NDCG@3
0,Popularity Baseline,0.095238,0.142857,0.142857
1,Content-Based,0.142857,0.285714,0.232990
2,Item-Based CF,0.142857,0.357143,0.224062


In [12]:
def catalogue_coverage(recommender, k=3):
    recommended = set()
    for user_id in train["user_id"].unique():
        recommended.update(recommender(user_id, k))
    return len(recommended) / movies["movie_id"].nunique()

coverage_results = pd.DataFrame({
    "Model": ["Popularity Baseline", "Content-Based", "Item-Based CF"],
    "Coverage": [
        catalogue_coverage(recommend_popularity),
        catalogue_coverage(recommend_content),
        catalogue_coverage(recommend_cf)
    ]
})

display(coverage_results)

,Model,Coverage
0,Popularity Baseline,0.35
1,Content-Based,0.85
2,Item-Based CF,0.80


In [13]:
pop_counts = train["movie_id"].value_counts().to_dict()
num_users = train["user_id"].nunique()

def novelty_of_item(movie_id):
    p = pop_counts.get(movie_id, 1) / num_users
    return -np.log2(p)

def average_novelty(recommender, k=3):
    values = []
    for user_id in train["user_id"].unique():
        recs = recommender(user_id, k)
        values.extend([novelty_of_item(m) for m in recs])
    return np.mean(values)

novelty_results = pd.DataFrame({
    "Model": ["Popularity Baseline", "Content-Based", "Item-Based CF"],
    "Novelty": [
        average_novelty(recommend_popularity),
        average_novelty(recommend_content),
        average_novelty(recommend_cf)
    ]
})

display(novelty_results)

,Model,Novelty
0,Popularity Baseline,1.386735
1,Content-Based,1.443840
2,Item-Based CF,1.677139


In [14]:
genre_similarity = pd.DataFrame(
    cosine_similarity(item_feature_matrix),
    index=item_feature_matrix.index,
    columns=item_feature_matrix.index
)

def list_diversity(movie_ids):
    sims = []
    for i in range(len(movie_ids)):
        for j in range(i + 1, len(movie_ids)):
            sims.append(genre_similarity.loc[movie_ids[i], movie_ids[j]])
    return 1 - np.mean(sims) if sims else 0

def average_diversity(recommender, k=3):
    return np.mean([
        list_diversity(recommender(user_id, k))
        for user_id in train["user_id"].unique()
    ])

diversity_results = pd.DataFrame({
    "Model": ["Popularity Baseline", "Content-Based", "Item-Based CF"],
    "Diversity": [
        average_diversity(recommend_popularity),
        average_diversity(recommend_content),
        average_diversity(recommend_cf)
    ]
})

display(diversity_results)

,Model,Diversity
0,Popularity Baseline,0.833333
1,Content-Based,0.465482
2,Item-Based CF,0.827580


In [15]:
summary = (
    rating_results
    .merge(ranking_results, on="Model")
    .merge(coverage_results, on="Model")
    .merge(novelty_results, on="Model")
    .merge(diversity_results, on="Model")
)

display(summary)

,Model,MAE,RMSE,Precision@3,Recall@3,NDCG@3,Coverage,Novelty,Diversity
0,Popularity Baseline,0.769841,0.916010,0.095238,0.142857,0.142857,0.35,1.386735,0.833333
1,Content-Based,2.090278,2.261033,0.142857,0.285714,0.232990,0.85,1.443840,0.465482
2,Item-Based CF,0.708795,0.861592,0.142857,0.357143,0.224062,0.80,1.677139,0.827580
